# V4 Prior Optimization: Training the Dixon-Coles Models

This notebook connects to the `v4_historical_data.sqlite` database you just populated with `soccerdata`.
It splits the massive 7,171-match database by League, and feeds each league's history into our custom `continuous_dixon_coles_nll` Maximum Likelihood Estimator to calculate the optimal Attack ($\alpha$) and Defense ($\beta$) parameters for every team.

In [6]:
import sqlite3
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

# Ensure we can import from the backend directory regardless of cwd
sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd() / 'v4_backend'))
sys.path.append(str(Path.cwd().parent.parent / 'v4_backend'))

from dixon_coles_xg import train_dixon_coles_prior

possible_db_paths = [
    Path("../../v4_historical_data.sqlite"), # If running from v4_backend/notebooks/
    Path("v4_historical_data.sqlite"),       # If running from root
    Path("../v4_historical_data.sqlite")     # If running from v4_backend/
]

DB_PATH = next((p for p in possible_db_paths if p.exists()), None)
if not DB_PATH:
    print("❌ Could not find v4_historical_data.sqlite in expected locations.")
else:
    print(f"✅ Found database at: {DB_PATH.resolve()}")


✅ Found database at: C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\FIFA-WORLDCUP-PREDICTION\v4_historical_data.sqlite


## 1. Load and Prep the Historical Ledger
We need to calculate `days_ago` to apply the exponential time-decay weighting.

In [7]:
try:
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM matches_xg", conn)
    conn.close()
    print(f"✅ Successfully loaded {len(df)} matches from SQLite database.")
except Exception as e:
    print(f"❌ Failed to load database: {e}")
    df = pd.DataFrame()

if not df.empty:
    # Convert date strings to datetime objects
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Drop any rows where dates failed to parse
    df = df.dropna(subset=['date'])
    
    # Calculate days_ago for the time decay function
    # We use the most recent match in the dataset as "today"
    max_date = df['date'].max()
    df['days_ago'] = (max_date - df['date']).dt.days
    
    print(f"Data spans from {df['date'].min().date()} to {max_date.date()}.")


✅ Successfully loaded 7156 matches from SQLite database.
Data spans from 2021-08-06 to 2025-05-25.


## 2. Optimize the Parameters per League
We loop through the 5 Big Leagues, dynamically mapping the team names to numeric indices so the Scipy SLSQP optimizer can vector-map them efficiently.

In [8]:
v4_priors = {}

if not df.empty:
    leagues = df['league'].unique()
    
    for lg in leagues:
        print(f"\n{'='*50}")
        print(f"📈 Optimizing Parameters for: {lg}")
        print(f"{'='*50}")
        
        # Filter matches for this specific league
        df_lg = df[df['league'] == lg].copy()
        
        # Get unique teams in this league
        teams_list = sorted(list(set(df_lg['home_team'].unique()) | set(df_lg['away_team'].unique())))
        
        # Map string team names to integer indices (0 to N)
        team_to_idx = {team: idx for idx, team in enumerate(teams_list)}
        df_lg['home_idx'] = df_lg['home_team'].map(team_to_idx)
        df_lg['away_idx'] = df_lg['away_team'].map(team_to_idx)
        
        # Rename columns to match what `continuous_dixon_coles_nll` expects
        df_lg = df_lg.rename(columns={
            'home_xg': 'obs_xg_h',
            'away_xg': 'obs_xg_a',
            'home_goals': 'act_g_h',
            'away_goals': 'act_g_a'
        })
        
        try:
            # Run the SLSQP optimizer!
            prior_dict, meta_params = train_dixon_coles_prior(df_lg, teams_list)
            
            v4_priors[lg] = {
                "teams": prior_dict,
                "meta": meta_params
            }
            
            print(f"\n✅ {lg} Optimization Complete!")
            print(f"   Home Advantage (gamma): {meta_params['gamma_home_advantage']:.3f}")
            print(f"   Draw Correction (rho): {meta_params['rho_draw_correction']:.3f}")
            
            # Let's peek at the top 3 and bottom 3 attacking teams
            sorted_attacks = sorted(prior_dict.items(), key=lambda x: x[1]['alpha'], reverse=True)
            print(f"\n   Top 3 Attacks: {sorted_attacks[0][0]} ({sorted_attacks[0][1]['alpha']:.2f}), {sorted_attacks[1][0]} ({sorted_attacks[1][1]['alpha']:.2f}), {sorted_attacks[2][0]} ({sorted_attacks[2][1]['alpha']:.2f})")
            print(f"   Bot 3 Attacks: {sorted_attacks[-1][0]} ({sorted_attacks[-1][1]['alpha']:.2f}), {sorted_attacks[-2][0]} ({sorted_attacks[-2][1]['alpha']:.2f}), {sorted_attacks[-3][0]} ({sorted_attacks[-3][1]['alpha']:.2f})")
            
        except Exception as e:
            print(f"\n❌ Optimization failed for {lg}: {e}")



📈 Optimizing Parameters for: ENG-Premier League
Initiating Maximum Likelihood Estimation via SLSQP...
Optimization terminated successfully    (Exit mode 0)
            Current function value: 897.0119262959672
            Iterations: 48
            Function evaluations: 2726
            Gradient evaluations: 48

✅ ENG-Premier League Optimization Complete!
   Home Advantage (gamma): 1.204
   Draw Correction (rho): -0.002

   Top 3 Attacks: Liverpool (1.61), Manchester City (1.41), Arsenal (1.33)
   Bot 3 Attacks: Norwich (0.55), Watford (0.68), Southampton (0.70)

📈 Optimizing Parameters for: ESP-La Liga
Initiating Maximum Likelihood Estimation via SLSQP...
Optimization terminated successfully    (Exit mode 0)
            Current function value: 980.5441147719615
            Iterations: 43
            Function evaluations: 2355
            Gradient evaluations: 43

✅ ESP-La Liga Optimization Complete!
   Home Advantage (gamma): 1.341
   Draw Correction (rho): 0.024

   Top 3 Attacks: B

## 3. Save the V4 Priors
We save the finalized mathematically-pure parameters to a JSON file so they can be injected into the UI dashboard or a real-time betting script.

In [9]:
if v4_priors:
    out_path = Path("../v4_priors.json")
    with open(out_path, "w") as f:
        json.dump(v4_priors, f, indent=2)
    print(f"\n💾 All V4 Prior parameters successfully exported to {out_path}")



💾 All V4 Prior parameters successfully exported to ..\v4_priors.json


In [4]:
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

random.seed(7)
np.random.seed(7)

# ---- Config ---------------------------------------------------------------
FINANCIAL_YEAR = "2026-27"
MONTHS = [("April", 2026, 4), ("May", 2026, 5), ("June", 2026, 6),
          ("July", 2026, 7), ("August", 2026, 8), ("September", 2026, 9)]

# Per-property "personality" so the demo tells a story.
# emp/con     = average employees / contractors
# rwc_p       = chance of a Restricted Work Case per month
# mtc_l       = average Medical Treatment Cases per month
# fac_l       = average First Aid Cases per month
# obs_l       = average unsafe situations observed per month
# rect        = share of observed unsafe situations that get rectified
# sess        = (min, max) safety training sessions per month
# miss_extra  = extra chance of NOT submitting in a month (late reporter)
PROFILES = {
    "Sarova Mara Game Camp":                dict(emp=105, con=15, rwc_p=0.02, mtc_l=0.15, fac_l=1.0, obs_l=3, rect=0.95, sess=(3, 5), miss_extra=0.00),  # clean, small
    "Sarova Stanley":                       dict(emp=360, con=40, rwc_p=0.03, mtc_l=0.25, fac_l=1.5, obs_l=6, rect=0.98, sess=(5, 8), miss_extra=0.00),  # star performer
    "Sarova Panafric":                      dict(emp=295, con=35, rwc_p=0.08, mtc_l=0.50, fac_l=2.0, obs_l=5, rect=0.92, sess=(3, 6), miss_extra=0.00),  # one LTI -> amber
    "Sarova Whitesands Beach Resort & Spa": dict(emp=325, con=45, rwc_p=0.10, mtc_l=0.90, fac_l=2.5, obs_l=5, rect=0.90, sess=(3, 5), miss_extra=0.00),  # TRC-FR amber
    "Sarova Lion Hill Game Lodge":          dict(emp=130, con=20, rwc_p=0.05, mtc_l=0.30, fac_l=1.5, obs_l=5, rect=0.60, sess=(2, 4), miss_extra=0.20),  # poor rectification, late reporter
    "Sarova Shaba Game Lodge":              dict(emp=110, con=15, rwc_p=0.30, mtc_l=0.80, fac_l=3.0, obs_l=6, rect=0.75, sess=(1, 3), miss_extra=0.00),  # problem property (2 LTIs)
    "Sarova Taita Hills Game Lodge":        dict(emp=118, con=18, rwc_p=0.08, mtc_l=0.50, fac_l=2.0, obs_l=4, rect=0.90, sess=(2, 5), miss_extra=0.00),  # improving trend
    "Sarova Salt Lick Game Lodge":          dict(emp=123, con=18, rwc_p=0.05, mtc_l=0.30, fac_l=1.5, obs_l=4, rect=0.92, sess=(2, 5), miss_extra=0.30),  # chronic late reporter
    "Sarova Woodlands Hotel & Spa":         dict(emp=175, con=25, rwc_p=0.05, mtc_l=0.30, fac_l=1.5, obs_l=4, rect=0.93, sess=(3, 6), miss_extra=0.00),
}

# Group-wide improvement over the year (multiplies incident rates)
TREND = {"April": 1.3, "May": 1.2, "June": 1.05, "July": 0.95, "August": 0.8, "September": 0.7}

# Extra month-by-month multiplier for one property (Taita improves sharply)
PROP_TREND = {"Sarova Taita Hills Game Lodge": [2.5, 2.0, 1.4, 0.9, 0.5, 0.2]}

# Deterministic incidents so the story is the same every run.
# LTIs include fatalities per policy; fatalities stay 0 in this demo.
FORCED = {
    ("Sarova Shaba Game Lodge", "June"):   {"lti": 1},
    ("Sarova Shaba Game Lodge", "August"): {"lti": 1, "hipo": 1},
    ("Sarova Panafric", "May"):            {"lti": 1},
    ("Sarova Whitesands Beach Resort & Spa", "July"): {"hipo": 1},
}

# Chance a property does NOT submit in a month (drives the response tracker)
MISSING_PROB = {"April": 0.02, "May": 0.04, "June": 0.06,
                "July": 0.06, "August": 0.10, "September": 0.30}

# Fake submitters per property: a primary person and a backup (used ~15% of the time)
SUBMITTERS = {
    "Sarova Mara Game Camp": ["Wanjiru Kamau", "Brian Otieno"],
    "Sarova Stanley": ["Amina Hassan", "Peter Mwangi"],
    "Sarova Panafric": ["Grace Njeri", "Samuel Kiptoo"],
    "Sarova Whitesands Beach Resort & Spa": ["Fatuma Mohamed", "David Omondi"],
    "Sarova Lion Hill Game Lodge": ["Joyce Chebet", "Kevin Maina"],
    "Sarova Shaba Game Lodge": ["Hassan Abdi", "Lucy Wambui"],
    "Sarova Taita Hills Game Lodge": ["Daniel Mutua", "Esther Achieng"],
    "Sarova Salt Lick Game Lodge": ["Mercy Wairimu", "Isaac Kilonzo"],
    "Sarova Woodlands Hotel & Spa": ["James Kariuki", "Naomi Cheruiyot"],
}
EMAIL_DOMAIN = "sarova-demo.example"  # reserved fake domain, can't reach real inboxes


def make_email(full_name):
    first, last = full_name.lower().split(" ", 1)
    return f"{first}.{last.replace(' ', '')}@{EMAIL_DOMAIN}"


INITIATIVES = [
    "One monthly safety committee meeting and {n} health and safety training sessions held. {i} workplace inspections completed.",
    "Weekly health talks held with staff. {n} safety trainings conducted and {i} inspections and one evaluation meeting done.",
    "Fire drill and evacuation exercise carried out. {n} safety trainings held; {i} inspections completed within the month.",
    "Monthly committee meeting held. {n} first aid and manual handling refresher sessions conducted. {i} inspections done.",
    "Safety toolbox talks held at start of shifts. {n} training sessions delivered and {i} inspections closed out.",
]

COLUMNS = [
    "Id", "Start time", "Completion time", "Email", "Name", "Property",
    "Financial Year", "Reporting Month", "Submitted by",
    "Safety Training Sessions", "Safety Training Participants",
    "Unsafe Situations Observed", "Unsafe Situations Rectified",
    "Health Awareness Sessions",
    "Fatalities",  # if your form column still has a trailing space ("Fatalities "), rename it
    "Major Fires", "Lost Time Injuries", "Restricted Work Cases",
    "Medical Treatment Cases", "First Aid Cases", "Total Manhours",
    "HIPO Incidents", "Road Related Incidents",
    "Average Employees", "Average Contractors",
    "Employees Injured Permanent Disability", "Minor Fires", "Guests Injured",
    "Incident Report", "Safety and Health Initiatives",
]

# ---- Generate ---------------------------------------------------------------
rows = []
rid = 1
for m_idx, (month, year, mnum) in enumerate(MONTHS):
    for prop, p in PROFILES.items():
        forced = FORCED.get((prop, month), {})

        # skip = property did not submit (never skip a month with a forced incident)
        miss = min(0.9, MISSING_PROB[month] + p["miss_extra"])
        if not forced and random.random() < miss:
            continue

        mult = TREND[month] * PROP_TREND.get(prop, [1] * 6)[m_idx]

        person = SUBMITTERS[prop][0] if random.random() < 0.85 else SUBMITTERS[prop][1]
        email = make_email(person)

        emp = max(10, int(np.random.normal(p["emp"], p["emp"] * 0.03)))
        con = max(0, int(np.random.normal(p["con"], max(2, p["con"] * 0.15))))
        manhours = int((emp + con) * 26 * 8 * np.random.uniform(0.95, 1.04))

        sessions = int(np.random.randint(p["sess"][0], p["sess"][1] + 1))
        participants = int(sessions * np.random.randint(10, 26))
        observed = int(np.random.poisson(p["obs_l"]))
        rectified = int(np.random.binomial(observed, p["rect"])) if observed else 0
        health = int(np.random.randint(1, 6))

        fatalities = 0
        lti = forced.get("lti", 0)
        rwc = int(np.random.binomial(1, min(1.0, p["rwc_p"] * mult)))
        mtc = int(np.random.poisson(p["mtc_l"] * mult))
        fac = int(np.random.poisson(p["fac_l"] * mult))
        hipo = forced.get("hipo", int(np.random.binomial(1, 0.05)))
        road = int(np.random.binomial(1, 0.05))
        perm_disability = 0
        minor_fires = int(np.random.binomial(1, 0.06))
        major_fires = 0
        guests_injured = int(np.random.poisson(0.3))

        insp = int(np.random.randint(1, 4))
        text = random.choice(INITIATIVES).format(n=sessions, i=insp)

        nm_year, nm_month = (year + 1, 1) if mnum == 12 else (year, mnum + 1)
        start = datetime(nm_year, nm_month, int(np.random.randint(1, 9)),
                         int(np.random.randint(8, 17)), int(np.random.randint(0, 60)))
        end = start + timedelta(minutes=int(np.random.randint(6, 25)))

        rows.append([
            rid, start, end, email, person, prop,
            FINANCIAL_YEAR, month, "Demo import",
            sessions, participants, observed, rectified, health,
            fatalities, major_fires, lti, rwc, mtc, fac, manhours,
            hipo, road, emp, con, perm_disability, minor_fires, guests_injured,
            None, text,
        ])
        rid += 1

df = pd.DataFrame(rows, columns=COLUMNS)

# ---- Output -------------------------------------------------------------------
with pd.ExcelWriter("pseudo_safety_form_responses.xlsx", engine="openpyxl",
                    datetime_format="yyyy-mm-dd hh:mm:ss") as writer:
    df.to_excel(writer, index=False, sheet_name="Form Responses")

df.to_csv("pseudo_safety_form_responses.csv", index=False)

# ---- Quick check: does the story land on the demo targets? --------------------
g = df.groupby("Property").agg(
    submissions=("Id", "count"),
    hours=("Total Manhours", "sum"),
    LTIs=("Lost Time Injuries", "sum"),
    RWCs=("Restricted Work Cases", "sum"),
    MTCs=("Medical Treatment Cases", "sum"),
    observed=("Unsafe Situations Observed", "sum"),
    rectified=("Unsafe Situations Rectified", "sum"),
)
g["TRC"] = g["LTIs"] + g["RWCs"] + g["MTCs"]
g["LTI-FR"] = (g["LTIs"] * 1_000_000 / g["hours"]).round(1)
g["TRC-FR"] = (g["TRC"] * 1_000_000 / g["hours"]).round(1)
g["Rectified %"] = (g["rectified"] / g["observed"] * 100).round(0)

print(f"\nCreated {len(df)} responses across {df['Property'].nunique()} properties\n")
print("Submissions per month:")
print(df.groupby("Reporting Month", sort=False)["Property"].count().to_string())
print("\nYTD by property (demo targets: LTI-FR <=2 green, >4 red | TRC-FR <=12 green, >20 red | Rectified >=90% green, <80% red):")
print(g[["submissions", "LTIs", "TRC", "LTI-FR", "TRC-FR", "Rectified %"]].to_string())


Created 48 responses across 9 properties

Submissions per month:
Reporting Month
April        9
May          9
June         9
July         8
August       7
September    6

YTD by property (demo targets: LTI-FR <=2 green, >4 red | TRC-FR <=12 green, >20 red | Rectified >=90% green, <80% red):
                                      submissions  LTIs  TRC  LTI-FR  TRC-FR  Rectified %
Property                                                                                 
Sarova Lion Hill Game Lodge                     4     0    3     0.0    25.2         52.0
Sarova Mara Game Camp                           6     0    2     0.0    13.5        100.0
Sarova Panafric                                 6     1    6     2.5    15.0         93.0
Sarova Salt Lick Game Lodge                     4     0    1     0.0     8.7         92.0
Sarova Shaba Game Lodge                         5     2   12    16.0    96.2         83.0
Sarova Stanley                                  6     0    3     0.0     6.1